# 01 · English STT benchmark (Colab GPU)

Runs every shortlisted speech-to-text model on the same clips and appends results to
`results.csv` in your Google Drive (`MyDrive/local-voice-research/results`), so a disconnect never loses finished work.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU` (or any GPU offered).
2. Key icon in the left sidebar → add secret **`HF_TOKEN`** with your Hugging Face token → switch on *Notebook access*.

**How to run:** run cells top to bottom. Each model section is independent.
If a section errors, screenshot the error and skip to the next section (30-minute rule).

## 1 · Setup (run this again after any *Restart session*)

In [ ]:
import os, sys, subprocess, importlib
from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

ROOT = "/content/drive/MyDrive/local-voice-research"
DATA, RESULTS = f"{ROOT}/data", f"{ROOT}/results"
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

if os.path.isdir("/content/lvr"):
    subprocess.run(["git", "-C", "/content/lvr", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Kiranrossi/local-voice-research", "/content/lvr"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisper_normalizer"])

sys.path.insert(0, "/content/lvr/src")
import harness as H; importlib.reload(H)
import gc, torch
import numpy as np
import transformers; transformers.logging.set_verbosity_error()  # hides repetitive generation warnings
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: set Runtime > Change runtime type > T4 GPU")
print("Text normaliser:", H.NORMALISER_NAME)

def run_all(model, name, sets=("librispeech", "svarah", "own_en", "noise"), **kw):
    """Runs one model on every test set that has been prepared. Returns {test_set: results row}."""
    rows = {}
    for ts in sets:
        m = f"{DATA}/{ts}/manifest.jsonl"
        if os.path.exists(m):
            rows[ts] = H.evaluate(model, name, m, ts, RESULTS, **kw)
        else:
            print(f"skip {ts}: not prepared yet")
    return rows

def hf_pipe(model_id, **kw):
    """Hugging Face ASR pipeline in fp16 on the GPU (works across transformers versions)."""
    from transformers import pipeline
    try:
        return pipeline("automatic-speech-recognition", model=model_id, dtype=torch.float16, device="cuda:0", **kw)
    except TypeError:  # older transformers
        return pipeline("automatic-speech-recognition", model=model_id, torch_dtype=torch.float16, device="cuda:0", **kw)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2 · Prepare test sets (once, about 5 minutes)

- **librispeech**: 100 random clips from LibriSpeech test-clean (sanity check against published numbers)
- **svarah**: 150 random clips of Indian-accented English (AI4Bharat Svarah, gated)
- **noise**: 10 clips with no speech (silence and noise), to measure hallucination

All clips are 1 to 30 s, saved as 16 kHz mono WAV in your Drive. Random sampling uses seed 42, so it is reproducible.

In [ ]:
import io, json
import numpy as np, soundfile as sf, librosa
from datasets import load_dataset, Audio

def save_clips(ds, name, n, min_s=1.0, max_s=30.0):
    out = f"{DATA}/{name}"
    if os.path.exists(f"{out}/manifest.jsonl"):
        print(f"{name}: already prepared"); return
    os.makedirs(out, exist_ok=True)
    audio_col = [k for k, f in ds.features.items() if isinstance(f, Audio)][0]
    text_col = next(k for k in ("text", "transcription", "sentence", "normalized_text") if k in ds.column_names)
    meta_cols = [k for k, f in ds.features.items()
                 if k not in (audio_col, text_col) and getattr(f, "dtype", None) == "string"]
    ds = ds.cast_column(audio_col, Audio(decode=False)).shuffle(seed=42)
    rows = []
    for ex in ds:
        a = ex[audio_col]
        raw = a["bytes"] if a.get("bytes") else open(a["path"], "rb").read()
        audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        dur = len(audio) / 16000
        if not (min_s <= dur <= max_s) or not ex[text_col]:
            continue
        cid = f"{name}_{len(rows):03d}"
        path = f"{out}/{cid}.wav"
        sf.write(path, audio, 16000, subtype="PCM_16")
        rows.append({"id": cid, "audio_filepath": path, "text": ex[text_col], "duration": round(dur, 3),
                     **{k: ex.get(k) for k in meta_cols}})
        if len(rows) >= n:
            break
    H.write_manifest(rows, f"{out}/manifest.jsonl")
    print(f"{name}: {len(rows)} clips, {sum(r['duration'] for r in rows)/60:.1f} min of audio")

# LibriSpeech test-clean (try the Open ASR Leaderboard copy first, then the original)
try:
    libri = load_dataset("hf-audio/esb-datasets-test-only-sorted", "librispeech", split="test.clean")
except Exception as e:
    print("ESB copy failed, trying openslr:", type(e).__name__)
    libri = load_dataset("openslr/librispeech_asr", "clean", split="test")
save_clips(libri, "librispeech", 100)

# Svarah (Indian-accented English)
svarah = load_dataset("ai4bharat/Svarah", split="test")
save_clips(svarah, "svarah", 150)

# No-speech clips for the hallucination test
if not os.path.exists(f"{DATA}/noise/manifest.jsonl"):
    H.make_noise_set(f"{DATA}/noise")
print("noise: 10 clips")

### Your own recordings

Upload the extracted `recordings` folder from the recorder ZIP to
`MyDrive/local-voice-research/data/recordings`, then run this cell.
It splits the clips into **own_en** (the 10 English sentences, used now) and **own_ta** (the 30 Tamlish sentences, used on Day 2).

In [ ]:
src = f"{DATA}/recordings/manifest.jsonl"
if not os.path.exists(src):
    print("Not found:", src, "\nUpload the 'recordings' folder to MyDrive/local-voice-research/data/ first.")
else:
    clips = H.read_manifest(src)  # makes paths absolute
    for name, prefix in (("own_en", "en_"), ("own_ta", "ta_")):
        part = [c for c in clips if c["id"].startswith(prefix)]
        H.write_manifest(part, f"{DATA}/{name}/manifest.jsonl")
        print(f"{name}: {len(part)} clips, {sum(c['duration'] for c in part):.0f} s of audio")
    missing = [c["id"] for c in clips if not os.path.exists(c["audio_filepath"])]
    print("Missing audio files:", missing or "none")

In [ ]:
# Quick look at the Svarah sample (who is speaking)
import pandas as pd
sv = pd.read_json(f"{DATA}/svarah/manifest.jsonl", lines=True)
print(sv.columns.tolist())
for col in ("gender", "age-group", "primary_language", "native_language"):
    if col in sv:
        print(f"\n{col}:\n", sv[col].value_counts().head(10).to_string())

## 3 · Whisper large-v3-turbo (expected LibriSpeech WER about 2 to 3%)

If our LibriSpeech WER is far from that, the setup is wrong. Stop and send me the output.

In [ ]:
class HFWhisper:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id)
    def transcribe(self, clip):
        out = self.pipe({"raw": clip["audio"].copy(), "sampling_rate": 16000},
                        generate_kwargs={"language": "en", "task": "transcribe"})
        return out["text"]

model = HFWhisper("openai/whisper-large-v3-turbo")
rows = run_all(model, "whisper-large-v3-turbo", variant="809M", runtime="transformers")
if "svarah" in rows:
    print("\nHardest Svarah clips for Whisper:")
    H.worst_examples(RESULTS, rows["svarah"]["run_id"], n=3)
del model; free_gpu()

## 4 · Moonshine Streaming (medium 245M and tiny 34M)

In [ ]:
class HFMoonshine:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id)
    def transcribe(self, clip):
        audio = clip["audio"]
        pad = (-len(audio)) % 320  # the streaming frontend needs a whole number of frames; pads at most 20 ms of silence
        audio = np.pad(audio, (0, pad))
        max_tokens = int(clip["duration"] * 6.5) + 10  # model card advice: cap length to avoid repetition loops
        return self.pipe({"raw": audio, "sampling_rate": 16000},
                         generate_kwargs={"max_new_tokens": max_tokens})["text"]

for size, params in (("medium", "245M"), ("tiny", "34M")):
    model = HFMoonshine(f"moonshine-ai/moonshine-streaming-{size}")
    run_all(model, f"moonshine-streaming-{size}", variant=params, runtime="transformers",
            notes="audio zero-padded to a multiple of 320 samples")
    del model; free_gpu()

## 5 · Qwen3-ASR (0.6B and 1.7B)

Installs the `qwen-asr` package. If the install or the model errors, screenshot it and move on to section 6.

In [ ]:
!pip install -q -U qwen-asr

In [ ]:
from qwen_asr import Qwen3ASRModel

class QwenASR:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "m"):
            self.m = Qwen3ASRModel.from_pretrained(self.model_id, dtype=torch.float16, device_map="cuda:0",
                                                   max_new_tokens=256)
    def transcribe(self, clip):
        return self.m.transcribe(audio=clip["path"], language="English")[0].text

for size in ("0.6B", "1.7B"):
    model = QwenASR(f"Qwen/Qwen3-ASR-{size}")
    run_all(model, f"qwen3-asr-{size}", variant=size, runtime="qwen-asr")
    del model; free_gpu()

## 6 · NVIDIA Parakeet TDT 0.6B (v2 English, v3 multilingual)

1. Run the install cell (about 5 minutes).
2. **Runtime → Restart session**.
3. Run the **Setup** cell (section 1) again, then the cell below. Data and results are on Drive, so nothing is lost.

In [ ]:
!pip install -q "nemo_toolkit[asr]"

In [ ]:
import nemo.collections.asr as nemo_asr

class Parakeet:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "m"):
            self.m = nemo_asr.models.ASRModel.from_pretrained(self.model_id).eval()
    def transcribe(self, clip):
        try:
            out = self.m.transcribe([clip["path"]], batch_size=1, verbose=False)
        except TypeError:
            out = self.m.transcribe([clip["path"]], batch_size=1)
        if isinstance(out, tuple): out = out[0]
        h = out[0]
        return h.text if hasattr(h, "text") else str(h)

for v in ("v2", "v3"):
    model = Parakeet(f"nvidia/parakeet-tdt-0.6b-{v}")
    run_all(model, f"parakeet-tdt-0.6b-{v}", variant="600M", quantization="fp32", runtime="nemo")
    del model; free_gpu()

## 7 · IBM Granite Speech 5.0 470M TurboCTC (optional, experimental)

This model is very new and its loading code is not verified. If it errors, skip it: it will be evaluated on published evidence only.

In [ ]:
class HFGeneric:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id, trust_remote_code=True)
    def transcribe(self, clip):
        return self.pipe({"raw": clip["audio"].copy(), "sampling_rate": 16000})["text"]

model = HFGeneric("ibm-granite/granite-speech-5.0-470m-turboctc")
run_all(model, "granite-speech-5.0-470m-turboctc", variant="470M", runtime="transformers")
del model; free_gpu()

## 8 · Results so far

In [ ]:
import pandas as pd
df = pd.read_csv(f"{RESULTS}/results.csv")
df = df[df.task == "stt"]
cols = ["model", "test_set", "wer", "wer_ci_low", "wer_ci_high", "hallucination_rate",
        "rtf", "latency_median_s", "peak_mem_gb", "device_name"]
pd.set_option("display.width", 200)
print(df[cols].to_string(index=False))
print(f"\nResults file: {RESULTS}/results.csv  (upload it to the chat)")